# Demofile Calculate Categories
Demofile to calculate rank and interval for a region and day interactively. Not meant for to reproduce the results of this project but to show how some functions work.  
For easier calculation for multiple days and regions, please use the main.py or calculate_categories.py file.

In [27]:
import pandas as pd

import regex as re
from datetime import datetime

import calculate_categories as cc
from config import *

In [29]:
# Config:

region_name = "vor"
# must be one of:
# ["vor", "ooevv", "esg", "verbundlinie", "kaernterlinien", "salzburgverkehr", "vvt", "vmobil", "obb"]

# select day in 2024 in format YYYYMMDD
selected_day = 20240131

In [30]:
# load data

path = PATH_IN_GTFS +  GTFS_REGIONS[region_name]

stops = pd.read_csv(path + "/stops.txt", quotechar='"', sep=",")
stop_times = pd.read_csv(path + "/stop_times.txt", quotechar='"', sep=",")
trips = pd.read_csv(path + "/trips.txt", quotechar='"', sep=",")
routes = pd.read_csv(path + "/routes.txt", quotechar='"', sep=",")
calendar = pd.read_csv(path + "/calendar.txt", quotechar='"', sep=",")
calendar_dates = pd.read_csv(path + "/calendar_dates.txt", quotechar='"', sep=",")

display(stops.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:10005:0:1,Sieggraben Gemeindeamt,47.650112,16.379279,1050.0,NaN,Pat:41:10005,Level 0,1
1,at:41:10005:0:2,Sieggraben Gemeindeamt,47.650099,16.379405,1050.0,NaN,Pat:41:10005,Level 0,2
2,at:41:10018:0:1,Antau Kleine Zeile,47.774540,16.475264,1051.0,NaN,Pat:41:10018,Level 0,1
3,at:41:10018:0:2,Antau Kleine Zeile,47.774450,16.475336,1051.0,NaN,Pat:41:10018,Level 0,2
4,at:41:10019:0:1,Stöttera Ost,47.770036,16.464475,1051.0,NaN,Pat:41:10019,Level 0,1


In [31]:
# filter calendar and calendar_dates

# find weekday of selected day
date = datetime.fromisoformat(str(selected_day))
day_string = date.strftime("%A").lower()

# only keep services that run on the selected day and weekday
calendar_filtered = calendar[(calendar['start_date'] <= selected_day) & (calendar['end_date'] >= selected_day) & (calendar[day_string] == 1)].copy()

# select services from exceptions
added_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 1)]
removed_service = calendar_dates[(calendar_dates['date'] == selected_day) & (calendar_dates['exception_type'] == 2)]

display(calendar_filtered.head())

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
2,T0#10,1,1,1,1,1,0,0,20231210,20240328
3,T0#100,1,1,1,1,1,0,0,20231210,20240303
7,T0#104,1,1,1,1,1,0,0,20231210,20241214
9,T0#106,1,1,1,1,1,0,0,20231210,20240330
12,T0#109,1,1,1,1,1,0,0,20231210,20241214


In [32]:
# filter to keep only valid trips
trips_filtered = trips[trips['service_id'].isin(calendar_filtered['service_id'])]

# remove services with exception_type = 2 from calendar_dates
trips_filtered = trips_filtered[~trips_filtered["service_id"].isin(removed_service["service_id"])]

# add services with exception_type = 1 from calendar_dates
trips_full = pd.concat([trips_filtered, trips[trips["service_id"].isin(added_service["service_id"])]])

if trips_full.shape[0] == 0:
    print(f"No trips found for {state_name} on {selected_day}! Either no service is running on this day or the input data is incomplete/incorrect.")

display(trips_full.head())

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vor:100:,T0#2,1.T0.70-100-j24-1.6.H,70-100-j24-1.6.H,St. Pölten Landhaus Klangturm,NaN,0,10536.0
1,at:vor:100:,T0#2,10.T0.70-100-j24-1.1.H,70-100-j24-1.1.H,St. Pölten Staudratgasse,NaN,0,10565.0
2,at:vor:100:,T0#2,100.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10581.0
3,at:vor:100:,T0#2,101.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10582.0
4,at:vor:100:,T0#2,102.T0.70-100-j24-1.3.R,70-100-j24-1.3.R,St. Pölten Hauptbahnhof,NaN,1,10583.0


In [33]:
# merge trips and routes

# merge trips with routes information
routes_trips = pd.merge(trips_full, routes, on='route_id', how='left')

routes_trips = routes_trips[routes_trips['route_type'].isin(ROUTE_TYPE_TRANSLATION.keys())]

# translate route type
routes_trips['trip_short_name'] = routes_trips['trip_short_name'].astype('str')
routes_trips['rank'] = routes_trips.apply(lambda x: cc.detect_route_type(x['trip_short_name'], x['route_type']), axis=1)

#prepare stops and stop_times
stops_filtered = stops.copy()
stops_filtered['stop_id'] = stops_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

display(stops_filtered.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:41:10005,Sieggraben Gemeindeamt,47.650112,16.379279,1050.0,NaN,Pat:41:10005,Level 0,1
1,at:41:10005,Sieggraben Gemeindeamt,47.650099,16.379405,1050.0,NaN,Pat:41:10005,Level 0,2
2,at:41:10018,Antau Kleine Zeile,47.774540,16.475264,1051.0,NaN,Pat:41:10018,Level 0,1
3,at:41:10018,Antau Kleine Zeile,47.774450,16.475336,1051.0,NaN,Pat:41:10018,Level 0,2
4,at:41:10019,Stöttera Ost,47.770036,16.464475,1051.0,NaN,Pat:41:10019,Level 0,1


In [34]:
# keep only stop entry for parent station, if no parent station is given, keep a stop entry
stops_parents = stops_filtered[stops_filtered['stop_id'].str.startswith('Pat')].copy()
stops_parents['stop_id'] = stops_parents['stop_id'].apply(lambda x: x if x[0] != 'P' else x[1:])
stops_filtered = stops_filtered[stops_filtered['stop_id'].str.startswith('at') | stops_filtered['stop_id'].str.startswith('obb')]
stops_filtered = stops_filtered[~stops_filtered['stop_id'].isin(stops_parents['stop_id'])].drop_duplicates(subset=['stop_id'], keep='first')
# TODO: maybe filter out special stations e.g. obb_CP_80854 Wattens Sammelpunkt Bahnhofstraße MPREIS or Pat:42:99979_HoB
stops_filtered_final = pd.concat([stops_parents, stops_filtered])

stop_times_filtered = stop_times.copy()
stop_times_filtered = stop_times_filtered[stop_times_filtered['departure_time'].between('06:00:00', '20:00:00')]
stop_times_filtered = stop_times_filtered[stop_times_filtered['stop_id'].str.startswith('at')]
# TODO: check if Parent station is in stop_times and keep those
stop_times_filtered['stop_id'] = stop_times_filtered['stop_id'].astype(str).apply(
    lambda x: (
        re.match(r'^((?:[^:]*:){3})', x).group(1).rstrip(':')
        if re.match(r'^((?:[^:]*:){3})', x)
        else x
    )
)

# add trip and route information to stop_times
stop_times_trips = pd.merge(stop_times_filtered, routes_trips, on='trip_id', how='inner')

display(stop_times_trips.head())

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,route_id,...,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type,rank
0,10.T0.1-CAT-j24-1.1.H,10:07:00,10:07:00,at:49:743,1,NaN,2,2,0.00,at:vor:1917:,...,1-CAT-j24-1.1.H,Flughafen Wien Bahnhof,CAT 9029,0,NaN,6,CAT,Wien Mitte - Flughafen Wien (VIE),2,1
1,10.T0.1-CAT-j24-1.1.H,10:23:00,10:23:00,at:43:4708,2,NaN,2,2,19638.45,at:vor:1917:,...,1-CAT-j24-1.1.H,Flughafen Wien Bahnhof,CAT 9029,0,NaN,6,CAT,Wien Mitte - Flughafen Wien (VIE),2,1
2,109.T0.1-CAT-j24-1.2.R,06:08:00,06:08:00,at:43:4708,1,NaN,2,2,0.00,at:vor:1917:,...,1-CAT-j24-1.2.R,Wien Mitte-Landstraße,CAT 9012,1,NaN,6,CAT,Wien Mitte - Flughafen Wien (VIE),2,1
3,109.T0.1-CAT-j24-1.2.R,06:24:00,06:24:00,at:49:743,2,NaN,2,2,19638.45,at:vor:1917:,...,1-CAT-j24-1.2.R,Wien Mitte-Landstraße,CAT 9012,1,NaN,6,CAT,Wien Mitte - Flughafen Wien (VIE),2,1
4,11.T0.1-CAT-j24-1.1.H,10:37:00,10:37:00,at:49:743,1,NaN,2,2,0.00,at:vor:1917:,...,1-CAT-j24-1.1.H,Flughafen Wien Bahnhof,CAT 9031,0,NaN,6,CAT,Wien Mitte - Flughafen Wien (VIE),2,1


In [35]:
# calculate rank intervals
stop_times_grouped = stop_times_trips.groupby(['stop_id']).agg(rank=("rank", "min"), count=("rank", "count")).reset_index().copy()

# merge with stops and kepp only needed columns
stops_final = pd.merge(stops_filtered_final.drop(['zone_id', 'location_type', 'level_id', 'platform_code', 'parent_station'], axis=1), stop_times_grouped, on='stop_id', how='inner')

display(stops_final)

,stop_id,stop_name,stop_lat,stop_lon,rank,count
0,at:41:10005,Sieggraben Gemeindeamt,47.649966,16.379351,3,111
1,at:41:10018,Antau Kleine Zeile,47.774498,16.475309,3,36
2,at:41:10019,Stöttera Ost,47.769928,16.464412,3,37
3,at:41:10020,Zemendorf Volksschule,47.765448,16.454648,3,37
4,at:41:10021,Zemendorf Wr. Neustädter Str.,47.762356,16.451638,3,37
...,...,...,...,...,...,...
10609,at:49:627,Wien Kagran U,48.243299,16.433762,1,1932
10610,at:49:752,Wien Lassallestraße,48.223834,16.402267,3,379
10611,at:49:827,Wien Margaretengürtel,48.188865,16.342655,1,1055
10612,at:49:950,Wien Nußdorfer Straße,48.231500,16.353803,1,1327


In [ ]:
# aggregate stops and calculate rank and interval for all regions, some of this is not needed for one single region
# but shown here for demonstration purposes

agg = stops_final.groupby(['stop_id']).agg(rank=("rank", "min"), count=("count", "sum")).reset_index()

# all_regions = pd.merge(agg, all_regions[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left')
stops_full = pd.merge(stops_final[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], agg, on='stop_id', how='right')

# calculate intervals and station categories
stops_full["interval"] = stops_full["count"].apply(lambda x: 840 / (x/2) if x != 0 else 1680)
stops_full["category"] = stops_full.apply(lambda x: cc.lookup_category(x["interval"], x["rank"]), axis=1)

# drop duplicates arising slightly different coords for same stop
all_regions = stops_full.drop_duplicates(subset=['stop_id'], keep='first')

display(stops_full.head())

,stop_id,stop_name,stop_lat,stop_lon,rank,count,interval,category
0,at:41:10005,Sieggraben Gemeindeamt,47.649966,16.379351,3,111,15.135135,3
1,at:41:10018,Antau Kleine Zeile,47.774498,16.475309,3,36,46.666667,5
2,at:41:10019,Stöttera Ost,47.769928,16.464412,3,37,45.405405,5
3,at:41:10020,Zemendorf Volksschule,47.765448,16.454648,3,37,45.405405,5
4,at:41:10021,Zemendorf Wr. Neustädter Str.,47.762356,16.451638,3,37,45.405405,5


In [39]:
stops_full.to_csv(PATH_OUT_STOPS + f"{region_name}_{selected_day}.csv", index=False)